# SOC L2 Lab — Eventos de seguridad generados

Consulta desde Python la actividad benigna marcada `SOC-L2-Pxx` que produce `Invoke-SocLabLogGeneration.ps1`.
Pensado para ejecutarse en la compute instance de Azure ML dentro de **Microsoft Sentinel > Notebooks**, o localmente en VS Code.

> Requisitos: `pip install azure-monitor-query azure-identity pandas`. La identidad necesita el rol *Log Analytics Reader* sobre el workspace.


In [ ]:
# Instalación de librerías requeridas en Azure ML
%pip install -q azure-identity azure-monitor-query pandas matplotlib


In [ ]:
# Parametros del laboratorio. Sustituye WORKSPACE_ID si usas otro workspace.
WORKSPACE_ID = '6cdf51ac-a467-4b2f-9d87-4bd5f69a0bec'  # customerId de soc-l2-law
LOOKBACK_HOURS = 24

In [ ]:
from datetime import timedelta
import pandas as pd
from azure.identity import DefaultAzureCredential
from azure.monitor.query import LogsQueryClient, LogsQueryStatus

credential = DefaultAzureCredential()
client = LogsQueryClient(credential)

def run_kql(query):
    resp = client.query_workspace(WORKSPACE_ID, query, timespan=timedelta(hours=LOOKBACK_HOURS))
    if resp.status == LogsQueryStatus.PARTIAL:
        table = resp.partial_data[0]
    elif resp.status == LogsQueryStatus.SUCCESS:
        table = resp.tables[0]
    else:
        raise RuntimeError(resp)
    return pd.DataFrame(data=table.rows, columns=table.columns)

## Resumen de marcadores por tabla


In [ ]:
resumen = '''
let win = SecurityEvent | where EventID in (4625,4688,4720,4722,4724,4725,4726,4727,4728,4729,4730,4732,4733,4738,4776) and (CommandLine has 'SOC-L2' or TargetUserName startswith 'soc-l2-p02' or TargetUserName startswith 'SOCp15' or TargetUserName startswith 'SOCg15' or MemberName has 'SOCp15') | summarize Eventos=count() by Tabla='SecurityEvent';
let sys = Syslog | where SyslogMessage has 'SOC-L2-P04' or (ProcessName in ('sshd','su') and SyslogMessage has 'socp04') or (Facility=='local5' and SyslogMessage has 'soc_l2_p13') or (Facility=='local6' and SyslogMessage has 'SOC-L2-P11') | summarize Eventos=count() by Tabla='Syslog';
let app = AppServiceHTTPLogs | where UserAgent startswith 'SOC-L2-Lab' | summarize Eventos=count() by Tabla='AppServiceHTTPLogs';
let kv = AZKVAuditLogs | where RequestUri has 'soc-l2-p07' or OperationName has 'Secret' | summarize Eventos=count() by Tabla='AZKVAuditLogs';
let aks = AKSAuditAdmin | where RequestUri has 'soc-l2-p08' | summarize Eventos=count() by Tabla='AKSAuditAdmin';
union win, sys, app, kv, aks | sort by Eventos desc
'''
run_kql(resumen)

## P02 y P15 — Autenticacion y gestion de cuentas en Windows


In [ ]:
run_kql('''
SecurityEvent
| where EventID == 4625 and TargetUserName startswith 'soc-l2-p02'
| summarize Intentos=count() by Computer, TargetUserName, LogonType, IpAddress
| sort by Intentos desc
''')

In [ ]:
run_kql('''
SecurityEvent
| where EventID in (4720,4722,4724,4725,4726,4727,4728,4729,4730,4732,4733,4738)
| where TargetUserName startswith 'SOCp15' or TargetUserName startswith 'SOCg15' or MemberName has 'SOCp15'
| project TimeGenerated, Computer, EventID, Activity, SubjectUserName, TargetUserName, MemberName
| sort by TimeGenerated asc
''')

## P04/P04B — Linux Syslog y accesos SSH/su fallidos


In [ ]:
run_kql('''
Syslog
| where SyslogMessage has 'SOC-L2-P04' or (ProcessName in ('sshd','su') and SyslogMessage has 'socp04')
| project TimeGenerated, Computer, Facility, ProcessName, SyslogMessage
| sort by TimeGenerated desc
''')

## P05 / P07 / P08 — App Service, Key Vault y AKS


In [ ]:
run_kql('''
AppServiceHTTPLogs
| where UserAgent startswith 'SOC-L2-Lab'
| summarize Peticiones=count() by CsMethod, ScStatus, CsUriStem, CIp
| sort by Peticiones desc
''')

In [ ]:
run_kql('''
AKSAuditAdmin
| where RequestUri has 'soc-l2-p08'
| project TimeGenerated, Verb, RequestUri, Usuario=tostring(parse_json(tostring(User)).username)
| sort by TimeGenerated desc
''')